In [2]:
import pandas as pd
import numpy as np

clients_df = pd.read_csv('../data/Clients.csv')
property_df = pd.read_csv('../data/Properties.csv')

In [3]:
print("Client IDs:")
print(clients_df['client_id'].head(10))

print("\nProperty Client References:")
print(property_df['client_ref'].dropna().head(10))

Client IDs:
0    C0001
1    C0002
2    C0003
3    C0004
4    C0005
5    C0006
6    C0007
7    C0008
8    C0009
9    C0010
Name: client_id, dtype: str

Property Client References:
0    C0027
1    C0097
2    C0113
3    C0141
4    C0146
5    C0023
6    C0040
7    C0007
8    C0082
9    C0038
Name: client_ref, dtype: str


In [4]:
print("Unique clients:", clients_df['client_id'].nunique())
print("Unique property client references:", property_df['client_ref'].nunique())

Unique clients: 2000
Unique property client references: 2000


In [5]:
property_client_ids = set(property_df['client_ref'].dropna())
client_ids = set(clients_df['client_id'])

unmatched_ids = property_client_ids - client_ids

print("Unmatched property client references:", len(unmatched_ids))

Unmatched property client references: 0


In [6]:
property_df['client_ref'].value_counts().describe()

count    2000.0000
mean        3.6525
std         0.8397
min         3.0000
25%         3.0000
50%         4.0000
75%         4.0000
max        13.0000
Name: count, dtype: float64

integration + feature engineering

In [7]:
print("Property shape:", property_df.shape)
print("Client shape:", clients_df.shape)

Property shape: (10000, 9)
Client shape: (2000, 12)


In [8]:
merged_df = clients_df.merge(
    property_df,
    how = 'left',
    right_on = 'client_ref',
    left_on = 'client_id',
    
)

In [9]:
merged_df.shape

(7305, 21)

In [10]:
merged_df.head(10)

,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,...,referral_channel,listing_id,tower_number,transaction_date,unit_category,unit_number,floor_area_sqft,sale_price,listing_status,client_ref
0,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,90343,9,10-01-2024,Apartment,40,1090.32,"$351,419.29",Sold,C0001
1,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,4051,4,12-01-2024,Apartment,51,1608.84,"$496,266.41",Sold,C0001
2,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,150099,15,05-01-2025,Apartment,15,522.71,"$175,599.90",Sold,C0001
3,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,30432,3,12-01-2025,Apartment,50,713.67,"$223,479.12",Sold,C0001
4,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,150044,15,01-01-2024,Apartment,6,938.57,"$299,245.20",Sold,C0002
5,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,1045,1,02-01-2024,Apartment,45,756.21,"$248,525.12",Sold,C0002
6,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,90285,9,12-01-2024,Apartment,36,1582.79,"$505,127.63",Sold,C0002
7,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,200450,20,05-01-2025,Apartment,51,1062.33,"$336,892.39",Sold,C0002
8,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,30377,3,12-01-2025,Apartment,46,1599.81,"$451,305.59",Sold,C0002
9,C0003,Individual,Kale,Gay,04-07-1959,M,USA,California,Home,4,...,Agency,10104,1,07-01-2024,Apartment,13,719.47,"$216,874.97",Sold,C0003


In [11]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].info()

<class 'pandas.DataFrame'>
RangeIndex: 7305 entries, 0 to 7304
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   floor_area_sqft   7305 non-null   float64
 1   sale_price        7305 non-null   str    
 2   transaction_date  7305 non-null   str    
dtypes: float64(1), str(2)
memory usage: 322.9 KB


In [12]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].head()

,floor_area_sqft,sale_price,transaction_date
0,1090.32,"$351,419.29",10-01-2024
1,1608.84,"$496,266.41",12-01-2024
2,522.71,"$175,599.90",05-01-2025
3,713.67,"$223,479.12",12-01-2025
4,938.57,"$299,245.20",01-01-2024


In [13]:
merged_df['sale_price'] = (
    merged_df['sale_price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [14]:
merged_df['transaction_date'] = pd.to_datetime(
    merged_df['transaction_date'],
    format='mixed',
    dayfirst=False,
    errors='coerce'
)

In [15]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].info()

<class 'pandas.DataFrame'>
RangeIndex: 7305 entries, 0 to 7304
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   floor_area_sqft   7305 non-null   float64       
 1   sale_price        7305 non-null   float64       
 2   transaction_date  7305 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(2)
memory usage: 171.3 KB


In [16]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].head()

,floor_area_sqft,sale_price,transaction_date
0,1090.32,351419.29,2024-10-01
1,1608.84,496266.41,2024-12-01
2,522.71,175599.90,2025-05-01
3,713.67,223479.12,2025-12-01
4,938.57,299245.20,2024-01-01


In [17]:
merged_df[['sale_price', 'floor_area_sqft']].describe()

,sale_price,floor_area_sqft
count,7305.000000,7305.000000
mean,345072.000115,1141.147838
std,132053.768307,419.241645
min,97402.800000,410.710000
25%,232448.630000,782.150000
50%,330997.320000,1110.880000
75%,451237.670000,1501.640000
max,736652.270000,1957.160000


# Features

Property Count

In [18]:
property_count = (
    merged_df.groupby('client_id')['listing_id']
    .count()
    .reset_index(name='property_count')
)

property_count.head()

,client_id,property_count
0,C0001,4
1,C0002,5
2,C0003,5
3,C0004,6
4,C0005,13


Total Property Value

In [19]:
total_property_value = (
    merged_df.groupby('client_id')['sale_price']
    .sum()
    .reset_index(name='total_property_value')
)

total_property_value.head()

,client_id,total_property_value
0,C0001,1246764.72
1,C0002,1841095.93
2,C0003,1661457.59
3,C0004,1608263.51
4,C0005,3653385.38


 Average value of Properties of each client

In [20]:
average_property_value = (
    merged_df.groupby('client_id')['sale_price']
    .mean()
    .reset_index(name='average_property_value')
)

average_property_value.head()

,client_id,average_property_value
0,C0001,311691.180000
1,C0002,368219.186000
2,C0003,332291.518000
3,C0004,268043.918333
4,C0005,281029.644615


Average floor area

In [21]:
average_floor_area = (
    merged_df.groupby('client_id')['floor_area_sqft']
    .mean()
    .reset_index(name='average_floor_area')
)

average_floor_area.head()

,client_id,average_floor_area
0,C0001,983.885000
1,C0002,1187.942000
2,C0003,1058.110000
3,C0004,937.103333
4,C0005,927.296154


In [22]:
client_features = property_count.merge(
    total_property_value,
    on='client_id',
    how='left'
)

client_features.head()

,client_id,property_count,total_property_value
0,C0001,4,1246764.72
1,C0002,5,1841095.93
2,C0003,5,1661457.59
3,C0004,6,1608263.51
4,C0005,13,3653385.38


In [23]:
client_features = client_features.merge(
    average_property_value,
    on='client_id',
    how='left'
)

client_features = client_features.merge(
    average_floor_area,
    on='client_id',
    how='left'
)

In [24]:
client_features.head()

,client_id,property_count,total_property_value,average_property_value,average_floor_area
0,C0001,4,1246764.72,311691.180000,983.885000
1,C0002,5,1841095.93,368219.186000,1187.942000
2,C0003,5,1661457.59,332291.518000,1058.110000
3,C0004,6,1608263.51,268043.918333,937.103333
4,C0005,13,3653385.38,281029.644615,927.296154


In [25]:
client_features.shape

(2000, 5)

In [26]:
client_features.isnull().sum()

client_id                 0
property_count            0
total_property_value      0
average_property_value    0
average_floor_area        0
dtype: int64

In [30]:
clients_df['date_of_birth'] = pd.to_datetime(
    clients_df['date_of_birth'],
    format='mixed',
    dayfirst=False,
    errors='coerce'
)

reference_date = pd.Timestamp('2026-01-01')

clients_df['age'] = (
    (reference_date - clients_df['date_of_birth']).dt.days // 365
)

In [31]:
client_info = clients_df[
    [
        'client_id',
        'client_type',
        'gender',
        'country',
        'region',
        'acquisition_purpose',
        'satisfaction_score',
        'loan_applied',
        'referral_channel',
        'age'
    ]
]

client_info.head()

,client_id,client_type,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel,age
0,C0001,Individual,F,USA,California,Home,4,Yes,Website,57
1,C0002,Individual,M,USA,California,Home,1,No,Website,63
2,C0003,Individual,M,USA,California,Home,4,Yes,Agency,66
3,C0004,Individual,M,USA,California,Home,5,No,Website,66
4,C0005,Company,M,USA,California,Investment,5,No,Website,49


In [33]:
client_info.shape

(2000, 10)

In [35]:
client_info.describe()

,satisfaction_score,age
count,2000.000000,2000.00000
mean,3.029000,55.14700
std,1.413562,17.35572
min,1.000000,25.00000
25%,2.000000,40.00000
50%,3.000000,56.00000
75%,4.000000,70.00000
max,5.000000,94.00000


In [36]:
client_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   client_id            2000 non-null   str  
 1   client_type          2000 non-null   str  
 2   gender               2000 non-null   str  
 3   country              2000 non-null   str  
 4   region               2000 non-null   str  
 5   acquisition_purpose  2000 non-null   str  
 6   satisfaction_score   2000 non-null   int64
 7   loan_applied         2000 non-null   str  
 8   referral_channel     2000 non-null   str  
 9   age                  2000 non-null   int64
dtypes: int64(2), str(8)
memory usage: 239.8 KB


In [37]:
client_features.isnull().sum()

client_id                 0
property_count            0
total_property_value      0
average_property_value    0
average_floor_area        0
dtype: int64

In [38]:
clients_df.columns

Index(['client_id', 'client_type', 'first_name', 'last_name', 'date_of_birth',
       'gender', 'country', 'region', 'acquisition_purpose',
       'satisfaction_score', 'loan_applied', 'referral_channel', 'age'],
      dtype='str')

In [39]:
client_info = clients_df[
    [
        'client_id',
        'client_type',
        'gender',
        'country',
        'region',
        'acquisition_purpose',
        'satisfaction_score',
        'loan_applied',
        'referral_channel',
        'age'
    ]
]

client_info.head()

,client_id,client_type,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel,age
0,C0001,Individual,F,USA,California,Home,4,Yes,Website,57
1,C0002,Individual,M,USA,California,Home,1,No,Website,63
2,C0003,Individual,M,USA,California,Home,4,Yes,Agency,66
3,C0004,Individual,M,USA,California,Home,5,No,Website,66
4,C0005,Company,M,USA,California,Investment,5,No,Website,49


In [40]:
client_features = client_info.merge(
    property_count,
    on='client_id',
    how='left'
)

client_features = client_features.merge(
    total_property_value,
    on='client_id',
    how='left'
)

client_features = client_features.merge(
    average_property_value,
    on='client_id',
    how='left'
)

client_features = client_features.merge(
    average_floor_area,
    on='client_id',
    how='left'
)

In [45]:
client_features.head()

,client_id,client_type,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel,age,property_count,total_property_value,average_property_value,average_floor_area
0,C0001,Individual,F,USA,California,Home,4,Yes,Website,57,4,1246764.72,311691.180000,983.885000
1,C0002,Individual,M,USA,California,Home,1,No,Website,63,5,1841095.93,368219.186000,1187.942000
2,C0003,Individual,M,USA,California,Home,4,Yes,Agency,66,5,1661457.59,332291.518000,1058.110000
3,C0004,Individual,M,USA,California,Home,5,No,Website,66,6,1608263.51,268043.918333,937.103333
4,C0005,Company,M,USA,California,Investment,5,No,Website,49,13,3653385.38,281029.644615,927.296154


In [41]:
client_features.shape

(2000, 14)

In [42]:
client_features.columns

Index(['client_id', 'client_type', 'gender', 'country', 'region',
       'acquisition_purpose', 'satisfaction_score', 'loan_applied',
       'referral_channel', 'age', 'property_count', 'total_property_value',
       'average_property_value', 'average_floor_area'],
      dtype='str')

In [43]:
client_features.isnull().sum()

client_id                 0
client_type               0
gender                    0
country                   0
region                    0
acquisition_purpose       0
satisfaction_score        0
loan_applied              0
referral_channel          0
age                       0
property_count            0
total_property_value      0
average_property_value    0
average_floor_area        0
dtype: int64

In [48]:
client_features.dtypes

client_id                     str
client_type                   str
gender                        str
country                       str
region                        str
acquisition_purpose           str
satisfaction_score          int64
loan_applied                  str
referral_channel              str
age                         int64
property_count              int64
total_property_value      float64
average_property_value    float64
average_floor_area        float64
dtype: object

In [49]:
pd.set_option('display.float_format', '{:,.2f}'.format)

In [50]:
client_features.describe(
)

,satisfaction_score,age,property_count,total_property_value,average_property_value,average_floor_area
count,"2,000.00","2,000.00","2,000.00","2,000.00","2,000.00","2,000.00"
mean,3.03,55.15,3.65,"1,260,375.48","347,089.96","1,147.48"
std,1.41,17.36,0.84,"347,830.66","69,721.82",219.92
min,1.00,25.00,3.00,"463,611.95","154,537.32",564.01
25%,2.00,40.00,3.00,"1,025,238.01","295,807.68",984.95
50%,3.00,56.00,4.00,"1,220,893.17","341,523.38","1,129.18"
75%,4.00,70.00,4.00,"1,441,973.88","390,843.32","1,296.56"
max,5.00,94.00,13.00,"3,653,385.38","563,423.50","1,800.45"


Clustering

In [51]:
ml_features = client_features.drop(columns=['client_id'])

ml_features.head()

,client_type,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel,age,property_count,total_property_value,average_property_value,average_floor_area
0,Individual,F,USA,California,Home,4,Yes,Website,57,4,"1,246,764.72","311,691.18",983.88
1,Individual,M,USA,California,Home,1,No,Website,63,5,"1,841,095.93","368,219.19","1,187.94"
2,Individual,M,USA,California,Home,4,Yes,Agency,66,5,"1,661,457.59","332,291.52","1,058.11"
3,Individual,M,USA,California,Home,5,No,Website,66,6,"1,608,263.51","268,043.92",937.10
4,Company,M,USA,California,Investment,5,No,Website,49,13,"3,653,385.38","281,029.64",927.30


In [52]:
ml_features.dtypes

client_type                   str
gender                        str
country                       str
region                        str
acquisition_purpose           str
satisfaction_score          int64
loan_applied                  str
referral_channel              str
age                         int64
property_count              int64
total_property_value      float64
average_property_value    float64
average_floor_area        float64
dtype: object

In [53]:
for col in [
    'client_type',
    'gender',
    'country',
    'region',
    'acquisition_purpose',
    'loan_applied',
    'referral_channel'
]:
    print(f"\n{col}:")
    print(ml_features[col].nunique(), "unique values")


client_type:
2 unique values

gender:
2 unique values

country:
10 unique values

region:
57 unique values

acquisition_purpose:
2 unique values

loan_applied:
2 unique values

referral_channel:
3 unique values


In [54]:
categorical_columns = [
    'client_type',
    'gender',
    'country',
    'region',
    'acquisition_purpose',
    'loan_applied',
    'referral_channel'
]

encoded_features = pd.get_dummies(
    ml_features,
    columns=categorical_columns,
    drop_first=False,
    dtype=int
)

encoded_features.head()

,satisfaction_score,age,property_count,total_property_value,average_property_value,average_floor_area,client_type_Company,client_type_Individual,gender_F,gender_M,...,region_Western Australia,region_Wyoming,region_Zealand,acquisition_purpose_Home,acquisition_purpose_Investment,loan_applied_No,loan_applied_Yes,referral_channel_Agency,referral_channel_Client,referral_channel_Website
0,4,57,4,"1,246,764.72","311,691.18",983.88,0,1,1,0,...,0,0,0,1,0,0,1,0,0,1
1,1,63,5,"1,841,095.93","368,219.19","1,187.94",0,1,0,1,...,0,0,0,1,0,1,0,0,0,1
2,4,66,5,"1,661,457.59","332,291.52","1,058.11",0,1,0,1,...,0,0,0,1,0,0,1,1,0,0
3,5,66,6,"1,608,263.51","268,043.92",937.10,0,1,0,1,...,0,0,0,1,0,1,0,0,0,1
4,5,49,13,"3,653,385.38","281,029.64",927.30,1,0,0,1,...,0,0,0,0,1,1,0,0,0,1


In [55]:
encoded_features.shape

(2000, 84)

Standardize the features

In [56]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaled_features = scaler.fit_transform(encoded_features)

In [59]:
scaled_features.shape

(2000, 84)